# `bkmg` affinity check -- graph structure vs. `bk32`/`bk08`

Standalone notebook, just for checking the `bkmg` affinity (BANKSY lambda=0.8,
kernel built with the SAME `pca_weight_on_existing_topology` max_gap-bandwidth
function `ctxg` uses, on a topology fixed independently of that bandwidth step --
see `graph_generator.py`'s `bkmg` branch for the full fix). An earlier version of
`bkmg` derived both the kNN topology AND the max_gap bandwidth from the same
BANKSY-embedding distances in one pass, which produced a near-singleton graph
(58313/58423 Leiden "communities", purity ~0.001) on this dataset -- deletes any
stale cache of that broken graph before rebuilding.

Builds `bk32`, `bk08`, and `bkmg`, then compares: mean edge weight, cell-type
purity, niche purity, JOINT (cell_type, niche) purity, and Leiden community count
at several resolutions, against the real number of (cell type, niche) pairs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Remove any stale `bkmg` cache first, before anything else

Drive-sync lag can leave a stale copy of the OLD, broken `bkmg` graph
(near-singleton, 58313/58423 Leiden "communities") sitting around under a few
possible paths depending on where Colab's cwd actually resolves `./graphs` to.
Glob and remove all of them now, before mounting/installing/loading anything that
might otherwise pick one up.

In [ ]:
import os
import glob

_removed = []
for _pattern in [
    './graphs/*bkmg*',
    '/content/graphs/*bkmg*',
    '/content/drive/MyDrive/codes/interpretable-prototype/graphs/*bkmg*',
]:
    for _f in glob.glob(_pattern):
        os.remove(_f)
        _removed.append(_f)

print(f'Removed {len(_removed)} stale bkmg cache file(s):' if _removed else 'No stale bkmg cache files found.')
for _f in _removed:
    print(' ', _f)


In [ ]:
!pip install -q scarches faiss-cpu scib-metrics
!pip install -q pybanksy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install "numpy<2.3"

**IMPORTANT: restart the runtime now** (Runtime -> Restart session) before running the cells below.

In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import scipy.sparse as sp

from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import CODE_DIR, get_affinity_path
from interpretable_ssl.augmenters.graph_generator import generate_affinity
from interpretable_ssl.evaluation.niche_program_recovery import (
    compute_ground_truth, save_ground_truth, load_ground_truth,
)

DS_ID = 's28nsc'
CT_KEY = 'celltypes'
NICHE_KEY = 'niches_2D'
BATCH_KEY = DATASETS[DS_ID].get('batch_key')
K_GRAPH = 50

GT_PATH = os.path.join(CODE_DIR, 'files', 'celltype_niches_full_s28nsc.csv')

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.dpi'] = 150

## Load dataset + ground truth (no X_ctx needed -- bk32/bk08/bkmg only use .X + obsm['spatial'])

In [ ]:
ds_conf = DATASETS[DS_ID]
adata = sc.read_h5ad(ds_conf['path'])
print(adata)

if os.path.exists(GT_PATH):
    print(f'Loading cached ground truth from {GT_PATH}')
    ground_truth = load_ground_truth(GT_PATH)
else:
    ground_truth = compute_ground_truth(adata, CT_KEY, NICHE_KEY, min_pos=5, min_ctrl=20)
    save_ground_truth(ground_truth, GT_PATH)

n_target = len(ground_truth)
print(f'Target: {n_target} real (cell type, niche) pairs in ground truth')

## Shared metric helpers

In [ ]:
def weighted_label_purity(A, labels):
    """For each row of sparse graph A, the weighted fraction of its neighbor edge
    weight going to other rows sharing the same label. Purely a property of the
    graph + labels -- no model involved."""
    cats = pd.Categorical(labels)
    onehot = np.eye(len(cats.categories), dtype=np.float32)[cats.codes]
    neighbor_label_mass = np.asarray(A @ onehot)
    total = neighbor_label_mass.sum(axis=1)
    own = neighbor_label_mass[np.arange(len(labels)), cats.codes]
    return np.divide(own, total, out=np.zeros_like(total), where=total > 0)


def leiden_communities(A, resolution):
    ad_graph = sc.AnnData(np.zeros((A.shape[0], 1), dtype=np.float32))
    try:
        sc.tl.leiden(ad_graph, resolution=resolution, adjacency=A, flavor='igraph',
                     n_iterations=2, directed=False, key_added='leiden')
    except TypeError:
        sc.tl.leiden(ad_graph, resolution=resolution, adjacency=A, key_added='leiden')
    return ad_graph.obs['leiden'].nunique()


def graph_compare_row(gname, resolutions=(0.5, 1.0, 2.0, 4.0, 8.0)):
    """Mean edge weight, ct/niche/joint purity, and Leiden community count at a
    few resolutions for an already-built graphs[gname]."""
    A_cmp = sp.csr_matrix(graphs[gname])
    A_cmp.setdiag(0)
    A_cmp.eliminate_zeros()
    joint_label = (adata.obs[CT_KEY].astype(str) + '_' + adata.obs[NICHE_KEY].astype(str)).to_numpy()
    row = {
        'graph': gname,
        'mean_edge_weight': A_cmp.data.mean(),
        'ct_purity_mean': weighted_label_purity(A_cmp, adata.obs[CT_KEY].to_numpy()).mean(),
        'niche_purity_mean': weighted_label_purity(A_cmp, adata.obs[NICHE_KEY].to_numpy()).mean(),
        'joint_purity_mean': weighted_label_purity(A_cmp, joint_label).mean(),
    }
    for res in resolutions:
        row[f'leiden_r{res}'] = leiden_communities(A_cmp, res)
    return row

## Build `bk32`, `bk08`, `bkmg`

`bk32`/`bk08` load from cache if already built elsewhere; `bkmg` was just forced
to rebuild fresh above.

In [ ]:
graphs = {}

for gname in ['bk32', 'bk08', 'bkmg']:
    cache_path = get_affinity_path(DS_ID, adata.n_obs, k_neighbors=K_GRAPH, affinity_type=gname)
    if os.path.exists(cache_path):
        print(f'=== {gname}: loading cached graph from {cache_path} ===')
        with open(cache_path, 'rb') as f:
            graphs[gname] = pickle.load(f)
    else:
        print(f'=== {gname}: no cache at {cache_path}, building ===')
        graphs[gname] = generate_affinity(adata, k=K_GRAPH, bk=BATCH_KEY, affinity_type=gname)
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, 'wb') as f:
            pickle.dump(graphs[gname], f)
        print(f'Saved to {cache_path} for future runs.')

## Compare: mean edge weight, ct/niche/joint purity, Leiden community count

In [ ]:
graph_compare_rows = [graph_compare_row(g) for g in ['bk32', 'bk08', 'bkmg']]

print(f'Target: {n_target} real (cell type, niche) pairs\n')
pd.DataFrame(graph_compare_rows).set_index('graph').round(3)